In [1]:
from pathlib import Path

from ultralytics import YOLO

from libs.dataset import (
    anonymize_dataset,
    create_dataset,
    create_retinal_segmentation_testset,
    download_images,
)
from libs.plotting import plot_LayerCAM, plot_LayerCAM_image
from libs.train import predict_segmentations_folder, test, train
from libs.utils import find_root_repository


We performed a KFOLD on the dataset to evaluate the YOLO model. Now we just train the model on all the dataset.

In [ ]:
root = find_root_repository()
train_csv_path = str(root / "dataset/source/csv/sources_kfold_train.csv")
test_csv_path = str(root / "dataset/source/csv/sources_kfold_test.csv")
images_ref_path = str(root / "dataset/image_ref")

anonymize_dataset(train_csv_path)
download_images(train_csv_path, f"{images_ref_path}/train")

anonymize_dataset(test_csv_path)
download_images(test_csv_path, f"{images_ref_path}/test")

## Dataset

IF YOU NEVER DOWNLOAD LOCALY IMAGES + ANONYMISE IT YOU TO

In [2]:
train_csv_path = str(find_root_repository()/ "dataset/source/csv/sources_kfold_train.csv")
test_csv_path = str(find_root_repository()/ "dataset/source/csv/sources_kfold_test.csv")
output_train = str(find_root_repository()/ "dataset/main/train")
output_test = str(find_root_repository()/ "dataset/main/test")

create_dataset(train_csv_path, output_train, source_col="LOCAL_REF")
create_dataset(test_csv_path, output_test, source_col="LOCAL_REF")

,SOURCE,PATIENT,TYPE,SIDE,PATHO,LOCAL_REF
0,"""Y:\260113\260113_LAC_L_1\260113_LAC_L_1_HD\pn...",LAC,retine,L,NaN,C:\Users\Zarichne\Documents\EyeSideClassificat...
1,"""Y:\260113\260113_LAC_R_1\260113_LAC_R_1_HD\pn...",LAC,retine,R,NaN,C:\Users\Zarichne\Documents\EyeSideClassificat...
2,"""Y:\260119\260119_TRE_L_1\260119_TRE_L_1_HD\pn...",TRE,retine,L,NaN,C:\Users\Zarichne\Documents\EyeSideClassificat...
3,"""Y:\260119\260119_TRE_R_1\260119_TRE_R_1_HD\pn...",TRE,retine,R,NaN,C:\Users\Zarichne\Documents\EyeSideClassificat...
4,"""Y:\260120\260120_YOS_L_1\260120_YOS_L_1_HD\pn...",YOS,retine,L,NaN,C:\Users\Zarichne\Documents\EyeSideClassificat...
...,...,...,...,...,...,...
79,"""Y:\251110\251110_SAF_R_1\251110_SAF_R_1_HD\pn...",SAF,retine,R,NaN,C:\Users\Zarichne\Documents\EyeSideClassificat...
80,"""Y:\251110\251110_WAD_L_1\251110_WAD_L_1_HD\pn...",WAD,retine,L,NaN,C:\Users\Zarichne\Documents\EyeSideClassificat...
81,"""Y:\251110\251110_WAD_R_1\251110_WAD_R_1_HD\pn...",WAD,retine,R,NaN,C:\Users\Zarichne\Documents\EyeSideClassificat...
82,"""Y:\251110\251110_WIM_L_2\251110_WIM_L_2_HD\pn...",WIM,retine,L,NaN,C:\Users\Zarichne\Documents\EyeSideClassificat...


## Training and Test 

Train on all the dataset and evaluate (test) on new image. We don't need validation dataset because we already have found best parameters

In [3]:
config = {"model": "../models/yolo11n-cls.pt", 
            "optimizer": 'SGD',
            "batch": 16,
            "epochs": 100,
            "imgsz": 512}

dataset_path = str(find_root_repository() / "dataset/main")

model = train("eyeSideClassificationV1_nano", dataset_path, config)
test(model, dataset_path)

New https://pypi.org/project/ultralytics/8.4.203 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.198  Python-3.14.6 torch-2.15.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5090, 32607MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\Zarichne\Documents\EyeSideClassification\dataset\main, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mi

(ultralytics.utils.metrics.ClassifyMetrics object with attributes:
 
 confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002933DC4D490>
 curves: []
 curves_results: []
 fitness: 0.988095223903656
 keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
 results_dict: {'metrics/accuracy_top1': 0.976190447807312, 'metrics/accuracy_top5': 1.0, 'fitness': 0.988095223903656}
 save_dir: WindowsPath('C:/Users/Zarichne/Documents/EyeSideClassification/runs/classify/val-6')
 speed: {'preprocess': 0.9885511904680503, 'inference': 2.452766666660214, 'loss': 0.00032619048050990044, 'postprocess': 0.0006440476237826044}
 top1: 0.976190447807312
 top5: 1.0,
 [0.976190447807312,
  np.float64(0.9761904761904762),
  np.float64(0.9761904761904762),
  np.float64(0.9761904761904762)])

## Specific tests

In [ ]:
# model.predict("moment0ff.png", visualize=True)

r = model.predict("../dataset/main/test/right/0081.png", imgsz=512, device=0, verbose=False, visualize=True)[0]
pred = model.names[r.probs.top1]

print(f"pref: {pred}, conf: {r.probs.top1conf.item()}")

In [ ]:
def print_error(model, dataset_test: str, config: dict):
    
    def _predict_class(class_name):
        path = Path(dataset_test, class_name)
        
        if not path.exists():
            print(f"Class {class_name} doesn't exist ! {path}")
            return False
        
        results = model(path, imgsz=config["imgsz"], stream=True, verbose=False)
        for r in results:
            if model.names[r.probs.top1] != class_name:
                print(f"Failed, predicted: {model.names[r.probs.top1]}, ground truth: {class_name} (conf: {r.probs.top1conf}) (path: {r.path})")

        return True
    
    _predict_class("left")
    _predict_class("right")

config = {"model": "../models/yolo11n-cls.pt", 
            "optimizer": 'SGD',
            "batch": 32,
            "epochs": 100,
            "imgsz": 512}


model = YOLO(find_root_repository() / "other/weights/LateralityCLS.pt")

print_error(model, str(find_root_repository() / "dataset/main/test"), config)

## XIA

Some thing that could be interesting is to view the layer-CAM to see the region that model look at to take a decision

In [ ]:
model = YOLO(find_root_repository() / "other/weights/LateralityCLS.pt")

In [ ]:
plot_LayerCAM(model, str(find_root_repository() / "dataset/main/test/left"), "left")

In [ ]:
plot_LayerCAM(model, str(find_root_repository() / "dataset/main/test/left"), "right")

Code for the article

In [ ]:
model = YOLO(find_root_repository() / "other/weights/LateralityCLS.pt")
paths = [("left/0000.png" , "left" ),
         ("right/0009.png", "right"),
         ("left/0022.png" , "left" ),
         ("right/0013.png", "right"),
         ("right/0021.png", "right"),
         ("left/0042.png" , "left" ),
         ("right/0051.png", "right"),
         ("left/0074.png" , "left" )]

for img, gd in paths:
    plot_LayerCAM_image(model, str(find_root_repository() / f"dataset/main/test/{img}"), gd)

In [ ]:
model = YOLO(find_root_repository() / "other/weights/LateralityCLS.pt")

plot_LayerCAM_image(model, str(find_root_repository() / "dataset/main/test/right/0069.png"), "right")

## Experiment: Is model detect retinal's vessel and can based it's prediction on it

In [ ]:
create_retinal_segmentation_testset("dataset/source/txt/retina_measures.txt", 
      "dataset/source/csv/sources_kfold_train.csv",
      "dataset/testseg",
      "dataset/source/csv/testseg.csv")

In [ ]:
model = YOLO("weights/eyeSideClassification_nano.pt")
predict_segmentations_folder(model, "dataset/testseg", "dataset/source/csv/testseg.csv")

In [ ]:
model = YOLO("weights/eyeSideClassification.pt")
plot_LayerCAM_image(model, "dataset/testseg/0000.png", "left")
plot_LayerCAM_image(model, "dataset/moment0ff.png", "")

### Result:
- accuracy artery: 0.57
- accuracy vein: 0.69
- accuracy both on the same image: 0.63

We can't conclude that the classifier detect or can based its predictions 
on the vessel's retinian.

It's just prove that the classifier need other things like choroidian vessels
or optic disk information for a better accuracy

## Export

In [ ]:
# export to MATLAB ONNX (opset version 17 important !)
model = YOLO("weights/eyeSideClassification.pt")
model.export(format="onnx", opset=17)